In [1]:
import re
import torch
import sentencepiece as spm
import pandas as pd
from collections import Counter
from torch.utils.data import Dataset, DataLoader

FON_COL, FR_COL = "fon", "fr"

final = {
    n: pd.read_csv(f"content_clean/{n}.csv", encoding="utf-8")
    for n in ["train", "val", "test"]
}

sp = spm.SentencePieceProcessor(model_file="models/spm_fr.model")

In [2]:
TOKEN_RE = re.compile(r'[^\s.,;:!?()"«»]+|[.,;:!?()"«»]')
PAD, UNK = 0, 1

def split_words(s):
    return TOKEN_RE.findall(s)

# Vocabulaire de caractères : construit sur le train uniquement
train_words = [w for s in final["train"][FON_COL] for w in split_words(s)]
char_counts = Counter(c for w in train_words for c in w)
char2id = {c: i + 2 for i, c in enumerate(sorted(char_counts))}   # 0 = PAD, 1 = UNK
print("caractères distincts :", len(char2id))

def word_to_chars(word, max_len=20):
    ids = [char2id.get(c, UNK) for c in word[:max_len]]
    return ids + [PAD] * (max_len - len(ids))

# Contrôles
train_vocab = set(train_words)
for n in ["train", "val", "test"]:
    sents = [split_words(s) for s in final[n][FON_COL]]
    words = [w for sent in sents for w in sent]
    unk_chars = sum(c not in char2id for w in words for c in w) / sum(len(w) for w in words)
    print(f"{n:5s} | mots/phrase : moy {sum(map(len, sents)) / len(sents):.1f}, max {max(map(len, sents))}"
          f" | mots > 20 caractères : {sum(len(w) > 20 for w in words) / len(words):.2%}"
          f" | caractères inconnus : {unk_chars:.3%}"
          f" | mots absents du train : {sum(w not in train_vocab for w in words) / len(words):.1%}")

w = split_words(final["val"][FON_COL][0])[0]
print(repr(w), word_to_chars(w))

caractères distincts : 107
train | mots/phrase : moy 7.9, max 63 | mots > 20 caractères : 0.02% | caractères inconnus : 0.000% | mots absents du train : 0.0%
val   | mots/phrase : moy 11.4, max 56 | mots > 20 caractères : 0.02% | caractères inconnus : 0.001% | mots absents du train : 5.0%
test  | mots/phrase : moy 12.9, max 63 | mots > 20 caractères : 0.04% | caractères inconnus : 0.004% | mots absents du train : 5.4%
'ali' [32, 43, 40, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [3]:
import torch, random, math
from torch.utils.data import Dataset, DataLoader

MAX_SRC_W, MAX_TGT_T, MAX_WORD_LEN = 70, 80, 20
BOS, EOS = sp.bos_id(), sp.eos_id()
id2char = {i: c for c, i in char2id.items()}

class PairDataset(Dataset):
    def __init__(self, df):
        self.items, self.cut_src, self.cut_tgt = [], 0, 0
        for fo, fr in zip(df[FON_COL], df[FR_COL]):
            words = split_words(fo)
            ids = sp.encode(fr)
            self.cut_src += len(words) > MAX_SRC_W
            self.cut_tgt += len(ids) > MAX_TGT_T - 2
            words, ids = words[:MAX_SRC_W], ids[:MAX_TGT_T - 2]
            if not words:
                continue
            chars = [word_to_chars(w, MAX_WORD_LEN) for w in words]
            self.items.append((chars, [BOS] + ids + [EOS], fr))
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate(batch):
    chars, tgts, refs = zip(*batch)
    B, T, L = len(batch), max(len(c) for c in chars), MAX_WORD_LEN
    src = torch.full((B, T, L), PAD, dtype=torch.long)
    for i, c in enumerate(chars):
        src[i, :len(c)] = torch.tensor(c)
    src_len = torch.tensor([len(c) for c in chars])
    U = max(len(t) for t in tgts)
    tgt = torch.full((B, U), PAD, dtype=torch.long)
    for i, t in enumerate(tgts):
        tgt[i, :len(t)] = torch.tensor(t)
    return src, src_len, tgt, list(refs)

class BucketSampler:
    """Regroupe les phrases de longueurs proches pour limiter le padding."""
    def __init__(self, ds, batch_size, shuffle):
        self.ds, self.bs, self.shuffle = ds, batch_size, shuffle
    def __len__(self): return math.ceil(len(self.ds) / self.bs)
    def __iter__(self):
        idx = list(range(len(self.ds)))
        if self.shuffle: random.shuffle(idx)
        chunk, batches = self.bs * 50, []
        for i in range(0, len(idx), chunk):
            part = sorted(idx[i:i + chunk], key=lambda j: len(self.ds.items[j][0]))
            batches += [part[k:k + self.bs] for k in range(0, len(part), self.bs)]
        if self.shuffle: random.shuffle(batches)
        return iter(batches)

datasets = {n: PairDataset(final[n]) for n in ["train", "val", "test"]}
loaders = {n: DataLoader(datasets[n], batch_sampler=BucketSampler(datasets[n], 64, n == "train"),
                         collate_fn=collate) for n in datasets}

for n, ds in datasets.items():
    print(f"{n:5s} | {len(ds)} phrases | source tronquée : {ds.cut_src} | cible tronquée : {ds.cut_tgt}")

# Contrôle d'intégrité sur un batch
src, src_len, tgt, refs = next(iter(loaders["train"]))
print("\nsrc :", tuple(src.shape), "| src_len :", src_len[:5].tolist(), "| tgt :", tuple(tgt.shape))
mot = "".join(id2char[i] for i in src[0, 0].tolist() if i > 1)
cible = sp.decode([i for i in tgt[0].tolist() if i not in (PAD, BOS, EOS)])
print("premier mot reconstitué :", repr(mot))
print("cible reconstituée      :", repr(cible))
print("référence d'origine     :", repr(refs[0]))

train | 27730 phrases | source tronquée : 0 | cible tronquée : 95
val   | 2242 phrases | source tronquée : 0 | cible tronquée : 18
test  | 1856 phrases | source tronquée : 0 | cible tronquée : 20

src : (64, 3, 20) | src_len : [3, 3, 3, 3, 3] | tgt : (64, 19)
premier mot reconstitué : 'ɖó'
cible reconstituée      : 'au nom de dieu'
référence d'origine     : 'au nom de dieu'


In [4]:
import torch.nn as nn
import torch.nn.functional as F

class Highway(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.h, self.g = nn.Linear(d, d), nn.Linear(d, d)
    def forward(self, x):
        g = torch.sigmoid(self.g(x))
        return g * F.relu(self.h(x)) + (1 - g) * x

class CharCNNEmbedding(nn.Module):
    """(B, T, L) indices de caractères -> (B, T, out_dim) : un vecteur par mot."""
    def __init__(self, n_chars, char_dim=32, widths=(2, 3, 4), n_filters=64, out_dim=256, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(n_chars, char_dim, padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Conv1d(char_dim, n_filters, w) for w in widths])
        d = n_filters * len(widths)
        self.highway = Highway(d)
        self.proj = nn.Linear(d, out_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, chars):
        B, T, L = chars.shape
        x = self.emb(chars.reshape(B * T, L)).transpose(1, 2)               # (B*T, char_dim, L)
        x = torch.cat([F.relu(c(x)).max(dim=2).values for c in self.convs], dim=1)
        x = self.proj(self.highway(x))
        return self.drop(x.view(B, T, -1))

n_chars = len(char2id) + 2          # + PAD et UNK
char_cnn = CharCNNEmbedding(n_chars)
print("paramètres du CharCNN :", sum(p.numel() for p in char_cnn.parameters()))

char_cnn.eval()
with torch.no_grad():
    out = char_cnn(src)
    seul = char_cnn(src[0:1, 0:1])          # le même mot, tout seul
print("entrée :", tuple(src.shape), "-> sortie :", tuple(out.shape))
print("le vecteur d'un mot ne dépend que de ses caractères :",
      torch.allclose(out[0, 0], seul[0, 0], atol=1e-5))

paramètres du CharCNN : 145632
entrée : (64, 3, 20) -> sortie : (64, 3, 256)
le vecteur d'un mot ne dépend que de ses caractères : True


In [5]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device :", device)
CFG = dict(rnn_type="gru", d=256, dropout=0.3)      # "gru" pour A3


class Encoder(nn.Module):
    """CharCNN -> BiLSTM : lit la phrase FONGBÉ."""
    def __init__(self, n_chars, rnn_type, d, dropout):
        super().__init__()
        self.is_lstm = rnn_type == "lstm"
        self.embed = CharCNNEmbedding(n_chars, out_dim=d, dropout=dropout)
        self.rnn = (nn.LSTM if self.is_lstm else nn.GRU)(d, d, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)

    def forward(self, src, src_len):
        x = self.embed(src)                                           # (B, T, d)
        packed = pack_padded_sequence(x, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, hidden = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)           # (B, T, 2d)
        h, c = hidden if self.is_lstm else (hidden, hidden)           # (2, B, d)
        return self.drop(out), h, c


class BahdanauAttention(nn.Module):
    """À chaque morceau de français, décide quels mots fongbé regarder."""
    def __init__(self, enc_dim, dec_dim, att_dim):
        super().__init__()
        self.w_enc = nn.Linear(enc_dim, att_dim, bias=False)
        self.w_dec = nn.Linear(dec_dim, att_dim)
        self.v = nn.Linear(att_dim, 1, bias=False)

    def forward(self, s, enc_out, mask):
        e = self.v(torch.tanh(self.w_enc(enc_out) + self.w_dec(s).unsqueeze(1))).squeeze(-1)
        e = e.masked_fill(~mask, float("-inf"))                       # ignorer le padding
        a = torch.softmax(e, dim=-1)
        return torch.bmm(a.unsqueeze(1), enc_out).squeeze(1), a


class Decoder(nn.Module):
    """Écrit le FRANÇAIS, un morceau à la fois."""
    def __init__(self, vocab, rnn_type, d, dropout):
        super().__init__()
        enc = 2 * d
        self.is_lstm = rnn_type == "lstm"
        self.embed = nn.Embedding(vocab, d, padding_idx=PAD)
        self.att = BahdanauAttention(enc, d, d)
        self.cell = (nn.LSTMCell if self.is_lstm else nn.GRUCell)(d + enc, d)
        self.mix = nn.Linear(d + enc + d, d)
        self.proj = nn.Linear(d, vocab)
        self.drop = nn.Dropout(dropout)

    def step(self, y, state, enc_out, mask):
        h, c = state
        e = self.drop(self.embed(y))
        ctx, a = self.att(h, enc_out, mask)
        x = torch.cat([e, ctx], dim=-1)
        if self.is_lstm:
            h, c = self.cell(x, (h, c))
        else:
            h = self.cell(x, h); c = h
        o = torch.tanh(self.mix(torch.cat([h, ctx, e], dim=-1)))
        return self.proj(self.drop(o)), (h, c), a


class Seq2Seq(nn.Module):
    def __init__(self, n_chars, tgt_vocab, rnn_type="lstm", d=256, dropout=0.3):
        super().__init__()
        self.enc = Encoder(n_chars, rnn_type, d, dropout)
        self.bridge_h = nn.Linear(2 * d, d)
        self.bridge_c = nn.Linear(2 * d, d)
        self.dec = Decoder(tgt_vocab, rnn_type, d, dropout)

    def encode(self, src, src_len):
        out, h, c = self.enc(src, src_len)
        h0 = torch.tanh(self.bridge_h(torch.cat([h[0], h[1]], dim=-1)))   # résumé de la phrase
        c0 = torch.tanh(self.bridge_c(torch.cat([c[0], c[1]], dim=-1)))
        lengths = src_len.to(out.device)
        mask = torch.arange(out.size(1), device=out.device)[None, :] < lengths[:, None]
        return out, (h0, c0), mask

    def forward(self, src, src_len, tgt):
        """Teacher forcing : entrée tgt[:, :-1], cible tgt[:, 1:]."""
        out, state, mask = self.encode(src, src_len)
        logits = []
        for t in range(tgt.size(1) - 1):
            lg, state, _ = self.dec.step(tgt[:, t], state, out, mask)
            logits.append(lg)
        return torch.stack(logits, dim=1)                              # (B, U-1, vocab)


model = Seq2Seq(n_chars, sp.get_piece_size(), **CFG).to(device)
print("paramètres : total", f"{sum(p.numel() for p in model.parameters()):,}",
      "| encodeur", f"{sum(p.numel() for p in model.enc.parameters()):,}",
      "| décodeur", f"{sum(p.numel() for p in model.dec.parameters()):,}")

# ---- Formes des vecteurs ----
src_b, src_len_b, tgt_b, _ = next(iter(loaders["train"]))
src_b, tgt_b = src_b.to(device), tgt_b.to(device)
model.eval()
with torch.no_grad():
    ch = model.enc.embed(src_b)
    out, (h0, c0), mask = model.encode(src_b, src_len_b)
    logits = model(src_b, src_len_b, tgt_b)
print("\nentrée (caractères)          :", tuple(src_b.shape))
print("après CharCNN  (par mot)     :", tuple(ch.shape))
print("après BiLSTM   (mot+contexte):", tuple(out.shape))
print("résumé de la phrase          :", tuple(h0.shape))
print("sortie du décodeur (logits)  :", tuple(logits.shape))

# ---- Le même mot dans deux phrases différentes ----
w = next(w for w, _ in Counter(train_words).most_common(300) if len(w) >= 3)
sents = [s for s in final["train"][FON_COL] if w in split_words(s)][:2]
pos = [split_words(s).index(w) for s in sents]
items = [([word_to_chars(x, MAX_WORD_LEN) for x in split_words(s)], [BOS, EOS], s) for s in sents]
s2, l2, _, _ = collate(items)
s2 = s2.to(device)
with torch.no_grad():
    c2 = model.enc.embed(s2)
    o2 = model.enc(s2, l2)[0]
print(f"\nmot testé : {w!r}")
for s in sents: print("  phrase :", s)
print("ressemblance (1.0 = identique) après CharCNN :",
      round(F.cosine_similarity(c2[0, pos[0]], c2[1, pos[1]], dim=0).item(), 4))
print("ressemblance (1.0 = identique) après BiLSTM  :",
      round(F.cosine_similarity(o2[0, pos[0]][:256], o2[1, pos[1]][:256], dim=0).item(), 4))

device : cpu
paramètres : total 4,497,280 | encodeur 935,136 | décodeur 3,299,488

entrée (caractères)          : (64, 5, 20)
après CharCNN  (par mot)     : (64, 5, 256)
après BiLSTM   (mot+contexte): (64, 5, 512)
résumé de la phrase          : (64, 256)
sortie du décodeur (logits)  : (64, 16, 4000)

mot testé : 'tɔn'
  phrase : ɖo tàjì ɔ́, azɔ̌mɛ̀vi xɔ atɔngɔ́ mɛ̀ ɖo lìnsínmɛ̀ sɔ́ yì xɔ gùdó gùdó tɔn mɛ̀ ɖo lìnsínmɛ̀ ɔ́ kpó gàn azɔ̌mɛ̀xwé ɔ́ tɔn kpó ɖó tó yě tàwùn.
  phrase : hwezivɔ do zo zunkan gle lɛ mɛ tɔn
ressemblance (1.0 = identique) après CharCNN : 1.0
ressemblance (1.0 = identique) après BiLSTM  : 0.9666


In [6]:
import time, subprocess, sys
import torch.nn as nn, torch.nn.functional as F
try:
    from sacrebleu.metrics import CHRF
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"])
    from sacrebleu.metrics import CHRF

torch.manual_seed(0)
V = sp.get_piece_size()
print("device :", device, "(passe en GPU avant l'entraînement complet)" if device.type == "cpu" else "")

def ids_to_text(ids):
    out = []
    for i in ids:
        if i == EOS: break
        if i not in (PAD, BOS): out.append(i)
    return sp.decode(out)

@torch.no_grad()
def greedy_decode(model, src, src_len, max_len=80):
    model.eval()
    out, state, mask = model.encode(src, src_len)
    y = torch.full((src.size(0),), BOS, dtype=torch.long, device=src.device)
    done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    res = []
    for _ in range(max_len):
        lg, state, _ = model.dec.step(y, state, out, mask)
        y = lg.argmax(-1)
        res.append(y)
        done |= (y == EOS)
        if done.all(): break
    return [ids_to_text(r) for r in torch.stack(res, 1).tolist()]

# ---- 200 paires fixes (4 lots de 64) ----
fixed, n = [], 0
for s, l, t, _ in loaders["train"]:
    fixed.append((s.to(device), l, t.to(device)))
    n += s.size(0)
    if n >= 200: break
refs = [ids_to_text(r) for _, _, t in fixed for r in t.tolist()]
print(f"{n} paires | exemple de référence : {refs[0]!r}")

def evaluate():
    hyps = [h for s, l, _ in fixed for h in greedy_decode(m, s, l)]
    return CHRF().corpus_score(hyps, [refs]).score, hyps

# ---- Modèle neuf, sans dropout ----
m = Seq2Seq(n_chars, V, **{**CFG, "dropout": 0.0}).to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
t0 = time.time()
for ep in range(1, 201):
    m.train(); tot, ntok = 0.0, 0
    for s, l, t in fixed:
        logits = m(s, l, t)
        gold = t[:, 1:]
        loss = F.cross_entropy(logits.reshape(-1, V), gold.reshape(-1), ignore_index=PAD, label_smoothing=0.1)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        with torch.no_grad():   # loss "simple" (sans lissage), plus lisible
            tot += F.cross_entropy(logits.reshape(-1, V), gold.reshape(-1), ignore_index=PAD, reduction="sum").item()
            ntok += (gold != PAD).sum().item()
    avg = tot / ntok
    if ep == 1 or ep % 10 == 0:
        score, hyps = evaluate()
        print(f"epoch {ep:3d} | loss {avg:.3f} | chrF {score:5.1f} | {time.time()-t0:.0f}s")
    if avg < 0.1: break

score, hyps = evaluate()
print(f"\nFINAL : epoch {ep} | loss {avg:.3f} | chrF {score:.1f}")
for i in (0, 1, 2):
    print("réf :", refs[i]); print("hyp :", hyps[i]); print()

device : cpu (passe en GPU avant l'entraînement complet)
256 paires | exemple de référence : 'il pleut'
epoch   1 | loss 7.955 | chrF   0.0 | 3s
epoch  10 | loss 4.233 | chrF   6.3 | 23s
epoch  20 | loss 2.727 | chrF  10.4 | 47s
epoch  30 | loss 1.553 | chrF  13.8 | 69s
epoch  40 | loss 0.933 | chrF  24.9 | 92s
epoch  50 | loss 0.578 | chrF  56.8 | 114s
epoch  60 | loss 0.327 | chrF  89.5 | 137s
epoch  70 | loss 0.205 | chrF  98.1 | 159s
epoch  80 | loss 0.158 | chrF  99.8 | 185s
epoch  90 | loss 0.139 | chrF  99.9 | 212s
epoch 100 | loss 0.130 | chrF  99.9 | 240s
epoch 110 | loss 0.124 | chrF  99.9 | 268s
epoch 120 | loss 0.121 | chrF  99.9 | 295s
epoch 130 | loss 0.118 | chrF  99.9 | 322s
epoch 140 | loss 0.116 | chrF  99.9 | 346s
epoch 150 | loss 0.115 | chrF  99.9 | 366s
epoch 160 | loss 0.114 | chrF  99.9 | 385s
epoch 170 | loss 0.103 | chrF  99.8 | 403s
epoch 180 | loss 0.131 | chrF  99.8 | 423s
epoch 190 | loss 0.114 | chrF  99.9 | 446s
epoch 200 | loss 0.112 | chrF  99.9 | 471s

In [7]:
import time, torch, torch.nn.functional as F
print("threads par défaut :", torch.get_num_threads())
batch = next(iter(loaders["train"]))
for nt in [4, 6, 8, 10]:
    torch.set_num_threads(nt)
    m = Seq2Seq(n_chars, sp.get_piece_size(), **CFG).to(device)
    o = torch.optim.Adam(m.parameters(), lr=1e-3)
    s, l, t = batch[0].to(device), batch[1], batch[2].to(device)
    t0 = time.time()
    for _ in range(20):
        lg = m(s, l, t)
        loss = F.cross_entropy(lg.reshape(-1, lg.size(-1)), t[:, 1:].reshape(-1), ignore_index=PAD)
        o.zero_grad(); loss.backward(); o.step()
    print(f"{nt:2d} threads : {(time.time()-t0)/20:.2f} s/étape")

threads par défaut : 10
 4 threads : 0.35 s/étape
 6 threads : 0.32 s/étape
 8 threads : 0.31 s/étape
10 threads : 0.33 s/étape


In [9]:
import os, csv, time, random, subprocess, sys
import torch, torch.nn as nn, torch.nn.functional as F
try:
    from sacrebleu.metrics import CHRF
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"])
    from sacrebleu.metrics import CHRF

# ---------- Réglages ----------
RUN = "A3_gru_P3"                      # change ce nom pour repartir de zéro (ex. A3_gru_P3)
CKPT_DIR = f"./checkpoints/{RUN}"
NUM_THREADS = 8                        # mets le meilleur résultat de ton test de vitesse
MAX_EPOCHS, EVAL_EVERY, PATIENCE = 30, 500, 3
LR, CLIP, SMOOTH = 1e-3, 1.0, 0.1
QUICK_N = 1000                          # phrases pour le chrF rapide (baisse à 500 si l'évaluation est trop lente)
TIME_BUDGET_MIN = None                   # minutes max pour cette session (None = pas de limite)

torch.set_num_threads(NUM_THREADS)
device = torch.device("cpu")
os.makedirs(CKPT_DIR, exist_ok=True)
V = sp.get_piece_size()

def ids_to_text(ids):
    out = []
    for i in ids:
        if i == EOS: break
        if i not in (PAD, BOS): out.append(i)
    return sp.decode(out)

@torch.no_grad()
def greedy_decode(model, src, src_len, max_len=80):
    model.eval()
    out, state, mask = model.encode(src, src_len)
    y = torch.full((src.size(0),), BOS, dtype=torch.long, device=src.device)
    done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    res = []
    for _ in range(max_len):
        lg, state, _ = model.dec.step(y, state, out, mask)
        y = lg.argmax(-1); res.append(y); done |= (y == EOS)
        if done.all(): break
    return [ids_to_text(r) for r in torch.stack(res, 1).tolist()]

# ---------- Validation : tous les lots + un sous-ensemble fixe pour le chrF ----------
val_batches = [(s, l, t) for s, l, t, _ in loaders["val"]]
order = list(range(len(val_batches))); random.Random(0).shuffle(order)
quick, n = [], 0
for i in order:
    quick.append(val_batches[i]); n += val_batches[i][0].size(0)
    if n >= QUICK_N: break
quick_refs = [ids_to_text(r) for _, _, t in quick for r in t.tolist()]

@torch.no_grad()
def evaluate():
    model.eval()
    tot, ntok = 0.0, 0
    for s, l, t in val_batches:
        lg = model(s, l, t); gold = t[:, 1:]
        tot += F.cross_entropy(lg.reshape(-1, V), gold.reshape(-1), ignore_index=PAD, reduction="sum").item()
        ntok += (gold != PAD).sum().item()
    hyps = [h for s, l, _ in quick for h in greedy_decode(model, s, l)]
    return tot / ntok, CHRF().corpus_score(hyps, [quick_refs]).score

# ---------- Modèle, optimiseur, reprise éventuelle ----------
torch.manual_seed(0)
model = Seq2Seq(n_chars, V, **CFG).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
step, epoch, best_val, best_step, bad = 0, 0, float("inf"), 0, 0
last = f"{CKPT_DIR}/last.pt"
if os.path.exists(last):
    ck = torch.load(last, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    step, epoch, best_val, best_step, bad = ck["step"], ck["epoch"], ck["best_val"], ck["best_step"], ck["bad"]
    print(f"REPRISE à l'étape {step} (epoch {epoch}), meilleure val {best_val:.3f}")

def save_ckpt(path):
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), step=step, epoch=epoch,
                    best_val=best_val, best_step=best_step, bad=bad,
                    config=dict(CFG, n_chars=n_chars, vocab=V), char2id=char2id), path)

log_path = f"{CKPT_DIR}/log.csv"
if not os.path.exists(log_path):
    with open(log_path, "w", newline="") as f:
        csv.writer(f).writerow(["step", "epoch", "train_loss", "val_loss", "val_chrf", "session_min"])

n_par = sum(p.numel() for p in model.parameters())
print(f"threads : {NUM_THREADS} | paramètres : {n_par:,} | {len(loaders['train'])} lots/epoch | évaluation toutes les {EVAL_EVERY} étapes\n")

# ---------- Entraînement ----------
t0 = time.time(); stop = False; run_loss, run_n, n_sess = 0.0, 0, 0
while epoch < MAX_EPOCHS and not stop:
    epoch += 1; te = time.time(); model.train()
    for s, l, t, _ in loaders["train"]:
        lg = model(s, l, t); gold = t[:, 1:]
        loss = F.cross_entropy(lg.reshape(-1, V), gold.reshape(-1), ignore_index=PAD, label_smoothing=SMOOTH)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP); opt.step()
        step += 1; n_sess += 1; run_loss += loss.item(); run_n += 1

        if n_sess == 50:
            est = (time.time() - t0) / 50 * len(loaders["train"]) / 60
            print(f"[vitesse] 50 étapes en {time.time()-t0:.0f}s -> environ {est:.1f} min par epoch\n")

        if step % EVAL_EVERY == 0:
            te_eval = time.time()
            vl, vc = evaluate(); model.train()
            tl = run_loss / run_n; run_loss, run_n = 0.0, 0
            mins = (time.time() - t0) / 60
            improved = vl < best_val
            if improved: best_val, best_step, bad = vl, step, 0
            else: bad += 1
            if improved: save_ckpt(f"{CKPT_DIR}/best.pt")
            save_ckpt(last)
            with open(log_path, "a", newline="") as f:
                csv.writer(f).writerow([step, epoch, round(tl, 4), round(vl, 4), round(vc, 2), round(mins, 1)])
            tag = "* meilleur" if improved else f"(sans gain {bad}/{PATIENCE})"
            print(f"étape {step:5d} | epoch {epoch:2d} | train {tl:.3f} | val {vl:.3f} | chrF {vc:4.1f} | "
                  f"{mins:5.1f} min (éval {time.time()-te_eval:.0f}s) {tag}")
            if bad >= PATIENCE:
                print("-> early stopping"); stop = True; break
            if TIME_BUDGET_MIN and mins >= TIME_BUDGET_MIN:
                print("-> budget de temps atteint"); stop = True; break
    print(f"--- epoch {epoch} : {(time.time()-te)/60:.1f} min")

print(f"\nTERMINÉ | paramètres {n_par:,} | meilleur modèle : étape {best_step}, val loss {best_val:.3f} "
      f"| {(time.time()-t0)/60:.0f} min cette session")

REPRISE à l'étape 6000 (epoch 14), meilleure val 4.088
threads : 8 | paramètres : 4,497,280 | 434 lots/epoch | évaluation toutes les 500 étapes

[vitesse] 50 étapes en 38s -> environ 5.5 min par epoch

--- epoch 15 : 5.5 min
étape  6500 | epoch 16 | train 3.354 | val 4.071 | chrF 18.7 |   6.6 min (éval 16s) * meilleur
--- epoch 16 : 5.3 min
étape  7000 | epoch 17 | train 3.297 | val 4.078 | chrF 19.2 |  12.6 min (éval 14s) (sans gain 1/3)
--- epoch 17 : 5.8 min
étape  7500 | epoch 18 | train 3.248 | val 4.078 | chrF 19.8 |  19.5 min (éval 16s) (sans gain 2/3)
--- epoch 18 : 6.4 min
étape  8000 | epoch 19 | train 3.236 | val 4.068 | chrF 19.7 |  27.3 min (éval 17s) * meilleur
--- epoch 19 : 7.2 min
étape  8500 | epoch 20 | train 3.222 | val 4.066 | chrF 19.4 |  36.8 min (éval 16s) * meilleur
--- epoch 20 : 7.8 min
étape  9000 | epoch 21 | train 3.153 | val 4.079 | chrF 20.0 |  44.6 min (éval 17s) (sans gain 1/3)
--- epoch 21 : 7.2 min
--- epoch 22 : 7.0 min
étape  9500 | epoch 23 | trai

In [11]:
import unicodedata, re, hashlib
from pathlib import Path
import pandas as pd

FON_COL, FR_COL = "fon", "fr"
MAX_SRC_WORDS, MAX_TGT_WORDS = 50, 60


# ---------- 1) Normalisation du TEXTE (appliquée en premier, avant tout dédoublonnage) ----------
# Caractères parasites observés dans tes données (encodages mélangés)
CHAR_MAP = str.maketrans({
    "ε": "ɛ", "Ε": "Ɛ",            # epsilon grec -> ɛ latin ouvert
    "đ": "ɖ", "ð": "ɖ", "Đ": "Ɖ",  # d barré / eth -> ɖ
    "ɩ": "i", "ɑ": "a",            # iota latin / alpha latin -> i / a
    "‚": ",", "¬": "-",            # virgule basse, signe "non" -> , et -
    "‘": "'", "’": "'", "ʼ": "'", "`": "'",
    "“": '"', "”": '"', "«": '"', "»": '"',
    "…": "...", "–": "-", "—": "-",
})


def norm_text(s):
    s = unicodedata.normalize("NFD", str(s))
    s = s.replace("\u0306", "\u030c")                          # brève -> caron
    s = s.lower()                                              # AVANT la table de correspondance
    s = unicodedata.normalize("NFC", s.translate(CHAR_MAP))
    s = "".join(c for c in s if unicodedata.category(c) != "Cf")
    return re.sub(r"\s+", " ", s).strip()


# ---------- 2) Clé de comparaison (sert uniquement à détecter doublons/fuites) ----------
def match_key(s):
    """Insensible à la casse, aux tons/accents, à la ponctuation et aux espaces."""
    s = unicodedata.normalize("NFKD", norm_text(s))
    s = "".join(c for c in s if not unicodedata.category(c).startswith("M"))
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def clean_splits(dfs, verbose=True, fr_leak_min_words=4):
    log = lambda *a: print(*a) if verbose else None
    d = {}
    for n, df in dfs.items():
        df = df[[FON_COL, FR_COL]].dropna().copy()
        df[FON_COL] = df[FON_COL].map(norm_text)
        df[FR_COL] = df[FR_COL].map(norm_text)
        df = df[(df[FON_COL] != "") & (df[FR_COL] != "")]
        df["k_fon"] = df[FON_COL].map(match_key)
        df["k_fr"] = df[FR_COL].map(match_key)
        d[n] = df[(df.k_fon != "") & (df.k_fr != "")].reset_index(drop=True)
        log(f"{n:5s} après normalisation : {len(d[n])} lignes")

    # --- TRAIN : retirer uniquement les paires (fon, fr) identiques
    n0 = len(d["train"])
    d["train"] = d["train"].drop_duplicates(["k_fon", "k_fr"]).reset_index(drop=True)
    log(f"train : {n0 - len(d['train'])} paires dupliquées retirées")
    # + filtre de longueur
    nf = d["train"][FON_COL].str.split().str.len()
    nr = d["train"][FR_COL].str.split().str.len()
    ok = (nf <= MAX_SRC_WORDS) & (nr <= MAX_TGT_WORDS) & (nr / nf).between(0.3, 3.0)
    log(f"train : {int((~ok).sum())} lignes retirées (longueur/ratio)")
    d["train"] = d["train"][ok].reset_index(drop=True)

    # --- VAL : source absente du train, et sans doublon de source interne
    tr_src = set(d["train"].k_fon)
    v = d["val"]
    v = v[~v.k_fon.isin(tr_src)].drop_duplicates("k_fon")
    log(f"val   : {len(d['val'])} -> {len(v)}")
    d["val"] = v.reset_index(drop=True)

    # --- TEST : source absente du train ET de val, et sans doublon interne
    t = d["test"]
    t = t[~t.k_fon.isin(tr_src | set(d["val"].k_fon))].drop_duplicates("k_fon")
    log(f"test  : {len(d['test'])} -> {len(t)}")
    d["test"] = t.reset_index(drop=True)

    # --- Fuite côté CIBLE : même phrase française (>= N mots) que dans un ensemble « supérieur »
    #     (souvent la même phrase fon écrite avec une autre orthographe)
    if fr_leak_min_words:
        long_fr = lambda df: df.k_fr.str.split().str.len() >= fr_leak_min_words
        v, t = d["val"], d["test"]
        v = v[~(v.k_fr.isin(set(d["train"].k_fr)) & long_fr(v))]
        t = t[~(t.k_fr.isin(set(d["train"].k_fr) | set(v.k_fr)) & long_fr(t))]
        log(f"fuite FR (>= {fr_leak_min_words} mots) : val -> {len(v)}, test -> {len(t)}")
        d["val"], d["test"] = v.reset_index(drop=True), t.reset_index(drop=True)

    # --- Vérifications strictes
    S = {n: set(df.k_fon) for n, df in d.items()}
    assert not (S["train"] & S["val"]),  "fuite train/val (source)"
    assert not (S["train"] & S["test"]), "fuite train/test (source)"
    assert not (S["val"] & S["test"]),   "fuite val/test (source)"
    assert d["val"].k_fon.is_unique and d["test"].k_fon.is_unique, "doublons internes val/test"
    assert not d["train"].duplicated(["k_fon", "k_fr"]).any(), "paires dupliquées dans train"
    log("OK : aucune source commune entre ensembles, aucun doublon dans val/test")

    # Info : fuites côté cible (français) — à surveiller, non supprimées
    R = {n: set(df.k_fr) for n, df in d.items()}
    log(f"info : phrases FR communes train/val {len(R['train'] & R['val'])}, "
        f"train/test {len(R['train'] & R['test'])}, val/test {len(R['val'] & R['test'])}")
    return d


def save_splits(final, out_dir="content_clean"):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    for n, df in final.items():
        p = out / f"{n}.csv"
        df[[FON_COL, FR_COL]].to_csv(p, index=False, encoding="utf-8")
        sha = hashlib.sha256(p.read_bytes()).hexdigest()[:12]
        print(f"{n:5s} | {len(df):6d} lignes | empreinte {sha}")


# ---------- Utilisation dans le notebook ----------
# dfs = {"train": pd.read_csv(...), "val": pd.read_csv(...), "test": pd.read_csv(...)}
# final = clean_splits(dfs)
# save_splits(final)


In [13]:
import torch, torch.nn.functional as F
torch.set_num_threads(4)

CKPT = "./checkpoints/A3_gru_P3/best.pt"     # pour A2 : ./checkpoints/A2_lstm_P3/best.pt
BEAM, MAX_LEN = 4, 80
V = sp.get_piece_size()

# ---- Charger le MEILLEUR modèle (et non les poids de la fin de l'entraînement) ----
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
cfg = {k: ck["config"][k] for k in ("rnn_type", "d", "dropout")}
model = Seq2Seq(n_chars, V, **cfg); model.load_state_dict(ck["model"]); model.eval()
print(f"chargé : {cfg['rnn_type']} | étape {ck['step']} (meilleure val {ck['best_val']:.3f})")

# ---- Beam search pour une phrase ----
@torch.no_grad()
def beam_search(enc_out, state, mask, k=BEAM, max_len=MAX_LEN):
    enc, m = enc_out.expand(k, -1, -1), mask.expand(k, -1)
    h, c = state[0].expand(k, -1).contiguous(), state[1].expand(k, -1).contiguous()
    seqs = torch.full((k, 1), BOS, dtype=torch.long)
    scores = torch.zeros(k); scores[1:] = float("-inf")
    done = []
    for _ in range(max_len):
        lg, (h, c), _ = model.dec.step(seqs[:, -1], (h, c), enc, m)
        cand = (scores[:, None] + F.log_softmax(lg, -1)).view(-1)
        top, idx = cand.topk(k)
        bi, tok = idx // V, idx % V
        seqs = torch.cat([seqs[bi], tok[:, None]], 1)
        h, c, scores = h[bi], c[bi], top
        for j in (tok == EOS).nonzero().flatten().tolist():
            done.append((scores[j].item() / (seqs.size(1) - 1), seqs[j, 1:-1].tolist()))
        scores = scores.masked_fill(tok == EOS, float("-inf"))
        if len(done) >= k or torch.isinf(scores).all(): break
    if not done:
        done.append((0.0, seqs[scores.argmax(), 1:].tolist()))
    return max(done, key=lambda x: x[0])[1]

# ---- Traduction d'un texte ----
@torch.no_grad()
def translate(text, beam=True):
    text = norm_text(text)
    words = split_words(text)[:MAX_SRC_W]
    if not words:
        return "(phrase vide)"
    items = [([word_to_chars(w, MAX_WORD_LEN) for w in words], [BOS, EOS], text)]
    s, l, _, _ = collate(items)
    model.eval()
    if beam:
        enc_out, (h0, c0), mask = model.encode(s, l)
        return sp.decode(beam_search(enc_out, (h0, c0), mask))
    return greedy_decode(model, s, l)[0]

# ---- Essai sur la VALIDATION ----
for fon, fr in list(zip(final["val"][FON_COL], final["val"][FR_COL]))[:6]:
    print("fon :", fon); print("réf :", fr); print("hyp :", translate(fon)); print()
print("--- ta phrase ---")
print(translate("bo nuɖe tε wε"))

chargé : gru | étape 8500 (meilleure val 4.066)
fon : ali vɔvɔ ɖokpo lɛ vɔ yi trele.
réf : il y a aussitôt une autre voie en terre rouge.
hyp : les fleurs d'abords sont allés au bénin.

fon : jǐkpɛ̀n - kpɛ̀n
réf : papaye
hyp : d'une blanche

fon : amɔ̌ nǔ è kàn ajɔ wíwá lɛ́ɛ ká ko bɛ́ɛ toolo ǎ.
réf : mais les échanges commerciaux n'ont pas pour autant repris systématiquement.
hyp : mais il n'y a pas d'avoir pas encore encore encore à tout ce qu'il n'est pas encore mûr.

fon : xɔ́ntɔ́n cé ɖíè
réf : voici mon ami.
hyp : voici mon ami

fon : un ɖo asi wo kpodo vi gban
réf : j'ai dix femmes et j'ai trente enfants.
hyp : j'ai beaucoup d'enfants et d'enfants

fon : nukɔntɔ́ wegɔ́ ɔ́ kpénukún azɔ̌ ée nyí étɔ́n é wú káká yi vivɔnu.
réf : le vice-président assure ses fonctions jusqu'à la fin du mandat entamé.
hyp : il y a trois jours de cotonou, il y a déjà reçu de l'étaillé.

--- ta phrase ---
et qu'est


In [14]:
@torch.no_grad()
def translate(text, beam=True):
    text = norm_text(text)                                  # même normalisation que les données
    words = split_words(text)[:MAX_SRC_W]
    if not words:
        return "(phrase vide)"
    items = [([word_to_chars(w, MAX_WORD_LEN) for w in words], [BOS, EOS], text)]
    s, l, _, _ = collate(items)                             # même fonction que pour l'entraînement
    model.eval()
    if beam:
        enc_out, (h0, c0), mask = model.encode(s, l)
        return sp.decode(beam_search(enc_out, (h0, c0), mask))
    return greedy_decode(model, s, l)[0]

# 1) Quelques phrases de la VALIDATION, avec la référence pour comparer
for fon, fr in list(zip(final["val"][FON_COL], final["val"][FR_COL]))[:6]:
    print("fon :", fon)
    print("réf :", fr)
    print("hyp :", translate(fon))
    print()

# 2) Ta propre phrase (modifie le texte entre guillemets)
print("--- ta phrase ---")
print(translate("bo nuɖe tε wε"))

fon : ali vɔvɔ ɖokpo lɛ vɔ yi trele.
réf : il y a aussitôt une autre voie en terre rouge.
hyp : les fleurs d'abords sont allés au bénin.

fon : jǐkpɛ̀n - kpɛ̀n
réf : papaye
hyp : d'une blanche

fon : amɔ̌ nǔ è kàn ajɔ wíwá lɛ́ɛ ká ko bɛ́ɛ toolo ǎ.
réf : mais les échanges commerciaux n'ont pas pour autant repris systématiquement.
hyp : mais il n'y a pas d'avoir pas encore encore encore à tout ce qu'il n'est pas encore mûr.

fon : xɔ́ntɔ́n cé ɖíè
réf : voici mon ami.
hyp : voici mon ami

fon : un ɖo asi wo kpodo vi gban
réf : j'ai dix femmes et j'ai trente enfants.
hyp : j'ai beaucoup d'enfants et d'enfants

fon : nukɔntɔ́ wegɔ́ ɔ́ kpénukún azɔ̌ ée nyí étɔ́n é wú káká yi vivɔnu.
réf : le vice-président assure ses fonctions jusqu'à la fin du mandat entamé.
hyp : il y a trois jours de cotonou, il y a déjà reçu de l'étaillé.

--- ta phrase ---
et qu'est
